## Single project

In [ ]:
from vod.evaluation import Evaluation
import os

# Groundtruth data transformed from BEV to Camera frame KITTI.
gt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/kitti_gt_annos_2/gt_lidar_to_camera_labels_2'

# Detection data transformed from BEV to Camera frame KITTI.
# check if code is correct, gt vs. gt (100% results)
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/kitti_gt_annos_2/gt_lidar_to_camera_labels_2'

# check Pytorch FP32 pred [-pi/4...3pi/4] vs. gt
dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_pt_fp32'
# check TRT FP32 pred [-pi/4...3pi/4] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_fp32'
# check TRT FP16 pred [-pi/4...3pi/4] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_fp16'
# check TRT INT8 pred [-pi/4...3pi/4] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_int8'

# check pred [0...pi/2] vs. gt
#dt_path = '/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds_regularized/pred_lidar_to_camera_fp32_rgd'

# When the instance is created, the label locations are required.
evaluation = Evaluation(test_annotation_file=os.path.join(gt_path)) # here gtlabels

# Using the evaluate method, the model can be evaluated on the detection labels.
results = evaluation.evaluate(
    result_path=os.path.join(dt_path), # here detection labels
    current_class=[0, 1, 2], score_thresh=0.1,
    eval_roi=False) 

metrics = results['entire_area']

# BBox
# NOTE: Dependent on 3D calculation
# print("\n=== 2D BBox Metrics (IoU=0.7) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls}: Easy: {metrics[f'{cls}_bbox_easy_07']:.2f}, " + f"Moderate: {metrics[f'{cls}_bbox_mod_07']:.2f}, " + f"Hard: {metrics[f'{cls}_bbox_hard_07']:.2f}")
# print("\n=== 2D BBox Metrics (IoU=0.5) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls}: Easy: {metrics[f'{cls}_bbox_easy_05']:.2f}, " + f"Moderate: {metrics[f'{cls}_bbox_mod_05']:.2f}, " + f"Hard: {metrics[f'{cls}_bbox_hard_05']:.2f}")

# 3D 
# NOTE: Too inaccurate due to missing detector parameters
# print("\n=== 3D Metrics (IoU=0.7) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls} 3d AP: {metrics[f'{cls}_3d_easy_07']:.2f}, " + f"{metrics[f'{cls}_3d_mod_07']:.2f}, " + f"{metrics[f'{cls}_3d_hard_07']:.2f}\n" +
#     f"mAP = {(metrics[f'{cls}_3d_easy_07'] + metrics[f'{cls}_3d_mod_07'] + metrics[f'{cls}_3d_hard_07'])/3:.2f}")
# print("\n=== 3D Metrics (IoU=0.5) ===")
# for cls in ['Car', 'Pedestrian', 'Cyclist']:
#     print(f"{cls} 3d AP: {metrics[f'{cls}_3d_easy_05']:.2f}, " + f"{metrics[f'{cls}_3d_mod_05']:.2f}, " + f"{metrics[f'{cls}_3d_hard_05']:.2f}\n" +
#     f"mAP = {(metrics[f'{cls}_3d_easy_05'] + metrics[f'{cls}_3d_mod_05'] + metrics[f'{cls}_3d_hard_05'])/3:.2f}")

# BEV
print("\n=== BEV Metrics (IoU=0.5) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_05']:.2f}, " + f"{metrics[f'{cls}_bev_mod_05']:.2f}, " + f"{metrics[f'{cls}_bev_hard_05']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_05'] + metrics[f'{cls}_bev_mod_05'] + metrics[f'{cls}_bev_hard_05'])/3:.2f}")

print("\n=== BEV Metrics (IoU=0.7) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_07']:.2f}, " + f"{metrics[f'{cls}_bev_mod_07']:.2f}, " + f"{metrics[f'{cls}_bev_hard_07']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_07'] + metrics[f'{cls}_bev_mod_07'] + metrics[f'{cls}_bev_hard_07'])/3:.2f}")
    
# Table print
print(f"{'':<10} {'AP at IoU=0.5 ↑':^38} {'AP at IoU=0.7 ↑':^38}")
print(f"{'':<10} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9}")

for cls in ['Car', 'Pedestrian', 'Cyclist']:
    easy_05 = metrics[f"{cls}_bev_easy_05"]
    mod_05  = metrics[f"{cls}_bev_mod_05"]
    hard_05 = metrics[f"{cls}_bev_hard_05"]
    avg_05  = (easy_05 + mod_05 + hard_05) / 3

    easy_07 = metrics[f"{cls}_bev_easy_07"]
    mod_07  = metrics[f"{cls}_bev_mod_07"]
    hard_07 = metrics[f"{cls}_bev_hard_07"]
    avg_07  = (easy_07 + mod_07 + hard_07) / 3

    print(f"{cls:<10} {easy_05:^9.2f} {mod_05:^9.2f} {hard_05:^9.2f} {avg_05:^9.2f} {easy_07:^9.2f} {mod_07:^9.2f} {hard_07:^9.2f} {avg_07:^9.2f}")

# visualization source for Easy/Mod./Hard
# "Voting for Voting in Online Point Cloud Object Detection"

## Master thesis KITTI
### Testing KITTI default, densified, interpolated, upsampled predictions

In [ ]:
from vod.evaluation import Evaluation
import os

# Groundtruth data transformed from BEV to Camera frame (KITTI format)
gt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/gts'

# check if code is correct, gt vs. gt (100% results), ATTENTION set score_thresh=-1
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/gts'

# Detection data transformed from BEV to Camera frame (KITTI format)
# Pytorch FP32 predictions
dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds'
# Pytorch FP32 densified predcitions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds_densified'
# Pytorch FP32 interpolated predcitions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds_interpolated'
# Pytorch FP32 upsampled predcitions
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_kitti_bev_val/transformed_labels/preds_upsampled'

# When the instance is created, the label locations are required.
evaluation = Evaluation(test_annotation_file=os.path.join(gt_path))

# Using the evaluate method, the model can be evaluated on the detection labels.
results = evaluation.evaluate(
    result_path=os.path.join(dt_path), 
    current_class=[0, 1, 2], score_thresh=0.1, # set this to -1, if you test gt vs. gt
    eval_roi=False) 

metrics = results['entire_area']

# BEV
print("\n=== BEV Metrics (IoU=0.5) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_05']:.2f}, " + f"{metrics[f'{cls}_bev_mod_05']:.2f}, " + f"{metrics[f'{cls}_bev_hard_05']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_05'] + metrics[f'{cls}_bev_mod_05'] + metrics[f'{cls}_bev_hard_05'])/3:.2f}")

print("\n=== BEV Metrics (IoU=0.7) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_07']:.2f}, " + f"{metrics[f'{cls}_bev_mod_07']:.2f}, " + f"{metrics[f'{cls}_bev_hard_07']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_07'] + metrics[f'{cls}_bev_mod_07'] + metrics[f'{cls}_bev_hard_07'])/3:.2f}")
    
# Table print
print(f"{'':<10} {'AP at IoU=0.5 ↑':^38} {'AP at IoU=0.7 ↑':^38}")
print(f"{'':<10} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9}")

for cls in ['Car', 'Pedestrian', 'Cyclist']:
    easy_05 = metrics[f"{cls}_bev_easy_05"]
    mod_05  = metrics[f"{cls}_bev_mod_05"]
    hard_05 = metrics[f"{cls}_bev_hard_05"]
    avg_05  = (easy_05 + mod_05 + hard_05) / 3

    easy_07 = metrics[f"{cls}_bev_easy_07"]
    mod_07  = metrics[f"{cls}_bev_mod_07"]
    hard_07 = metrics[f"{cls}_bev_hard_07"]
    avg_07  = (easy_07 + mod_07 + hard_07) / 3

    print(f"{cls:<10} {easy_05:^9.2f} {mod_05:^9.2f} {hard_05:^9.2f} {avg_05:^9.2f} {easy_07:^9.2f} {mod_07:^9.2f} {hard_07:^9.2f} {avg_07:^9.2f}")

## Master thesis ZOD
### Testing Zenseact default predictions

In [ ]:
from vod.evaluation import Evaluation
import os

# Groundtruth data transformed from BEV to Camera frame (KITTI format)
gt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'

# check if code is correct, gt vs. gt (100% results), ATTENTION set score_thresh=-1
#dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/gts'

# Detection data transformed from BEV to Camera frame (KITTI format)
# Pytorch FP32 predictions
dt_path = '/home/heizung1/ultralytics_yolov8-obb_ob_kitti/ultralytics/master_thesis/mt_zod_bev_val/transformed_labels/preds'

# When the instance is created, the label locations are required.
evaluation = Evaluation(test_annotation_file=os.path.join(gt_path))

# Using the evaluate method, the model can be evaluated on the detection labels.
results = evaluation.evaluate(
    result_path=os.path.join(dt_path), 
    current_class=[0, 1, 2], score_thresh=0.1, # set this to -1, if you test gt vs. gt
    eval_roi=False) 

metrics = results['entire_area']

# BEV
print("\n=== BEV Metrics (IoU=0.5) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_05']:.2f}, " + f"{metrics[f'{cls}_bev_mod_05']:.2f}, " + f"{metrics[f'{cls}_bev_hard_05']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_05'] + metrics[f'{cls}_bev_mod_05'] + metrics[f'{cls}_bev_hard_05'])/3:.2f}")

print("\n=== BEV Metrics (IoU=0.7) ===")
for cls in ['Car', 'Pedestrian', 'Cyclist']:
    print(f"{cls} bev AP: {metrics[f'{cls}_bev_easy_07']:.2f}, " + f"{metrics[f'{cls}_bev_mod_07']:.2f}, " + f"{metrics[f'{cls}_bev_hard_07']:.2f}\n" +
    f"mAP = {(metrics[f'{cls}_bev_easy_07'] + metrics[f'{cls}_bev_mod_07'] + metrics[f'{cls}_bev_hard_07'])/3:.2f}")
    
# Table print
print(f"{'':<10} {'AP at IoU=0.5 ↑':^38} {'AP at IoU=0.7 ↑':^38}")
print(f"{'':<10} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9} {'Easy':^9} {'Moderate':^9} {'Hard':^9} {'Average':^9}")

for cls in ['Car', 'Pedestrian', 'Cyclist']:
    easy_05 = metrics[f"{cls}_bev_easy_05"]
    mod_05  = metrics[f"{cls}_bev_mod_05"]
    hard_05 = metrics[f"{cls}_bev_hard_05"]
    avg_05  = (easy_05 + mod_05 + hard_05) / 3

    easy_07 = metrics[f"{cls}_bev_easy_07"]
    mod_07  = metrics[f"{cls}_bev_mod_07"]
    hard_07 = metrics[f"{cls}_bev_hard_07"]
    avg_07  = (easy_07 + mod_07 + hard_07) / 3

    print(f"{cls:<10} {easy_05:^9.2f} {mod_05:^9.2f} {hard_05:^9.2f} {avg_05:^9.2f} {easy_07:^9.2f} {mod_07:^9.2f} {hard_07:^9.2f} {avg_07:^9.2f}")